In [1]:
# required library
!pip install sentence-transformers

In [3]:
# pip install transformers torch numpy

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()


def mean_pooling(last_hidden_state, attention_mask):
    """
    Mean-pool token embeddings while ignoring padding tokens.
    """
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def embed_texts_clinicalbert(
    texts,
    batch_size=8,
    max_length=512,
    pooling="mean"
):
    """
    Convert a list of clinical notes into ClinicalBERT embeddings.

    Args:
        texts (list[str]): raw clinical notes
        batch_size (int): batch size for inference
        max_length (int): max token length; BERT-style models typically cap at 512
        pooling (str): "mean" or "cls"

    Returns:
        np.ndarray: shape (n_texts, hidden_size)
    """
    all_embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden_state = outputs.last_hidden_state  # (batch, seq_len, hidden_size)

            if pooling == "mean":
                batch_embeddings = mean_pooling(last_hidden_state, attention_mask)
            elif pooling == "cls":
                batch_embeddings = last_hidden_state[:, 0, :]
            else:
                raise ValueError("pooling must be either 'mean' or 'cls'")

            all_embeddings.append(batch_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# mock clinical notes to test the embedding model output
texts = [
    "Patient is a 74-year-old male with CHF, CKD stage 3, and shortness of breath. Discharged on furosemide.",
    "Patient improved clinically, vitals stable, follow-up with cardiology arranged in two weeks."
]

embeddings = embed_texts_clinicalbert(texts, batch_size=2, max_length=512, pooling="mean")

print("Embeddings shape:", embeddings.shape)
print("First 5 values of first embedding:", embeddings[0][:5])

Embeddings shape: (2, 768)
First 5 values of first embedding: [-0.00501387 -0.10414125 -0.16426538  0.15757406  0.24417375]
